# Klasifikasi Alfabet BISINDO dengan CNN dan MobileNetV2
Proyek UAS Machine Learning: Pengembangan Aplikasi Web untuk Pengenalan Alfabet Bahasa Isyarat Indonesia (BISINDO) Secara Real-Time.

Notebook ini berisi seluruh bagian pemrosesan data dan pelatihan model:
1. Membaca dan mengecek dataset
2. Membagi data latih, validasi, dan uji berdasarkan perangkat (HP) yang dipakai memotret
3. Melatih CNN yang dibangun dari awal (baseline)
4. Melatih MobileNetV2 dengan transfer learning
5. Evaluasi: precision, recall, F1-score, akurasi, waktu pelatihan, dan waktu prediksi
6. Eksperimen k-fold cross validation untuk kedua model
7. Menyimpan model terbaik untuk dipakai di aplikasi web

Dataset: Mendeley Data, https://data.mendeley.com/datasets/ywnjpbcz8m/1 (folder `02. Resized Images`, 26 kelas huruf A sampai Z, ukuran 224 x 224 piksel).

## 1. Import Library dan Pengaturan

In [ ]:
# Jalankan sekali saja kalau library belum ada
# %pip install tensorflow scikit-learn pandas matplotlib seaborn openpyxl

import os
import re
import json
import time
import hashlib
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow:', tf.__version__)
print('GPU tersedia:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Lokasi dataset dan folder output
DATASET_DIR = Path('dataset/raw/02. Resized Images')
MODEL_DIR = Path('models')
RESULT_DIR = Path('results')
MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

# Pengaturan umum
SEED = 42
BATCH_SIZE = 32
MNV2_SIZE = 224        # ukuran input MobileNetV2 (sesuai bobot ImageNet)
CNN_SIZE = 128         # CNN dari awal memakai ukuran lebih kecil supaya pelatihan lebih ringan

# Jumlah epoch
CNN_EPOCHS = 40
MNV2_HEAD_EPOCHS = 5
MNV2_FT_EPOCHS = 15

# Eksperimen k-fold (paling lama, bisa dimatikan dulu kalau mau cepat)
RUN_KFOLD = True
K = 3
KFOLD_CNN_EPOCHS = 25
KFOLD_MNV2_FT_EPOCHS = 10

# Seed supaya hasil bisa diulang
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Membaca Dataset
Setiap subfolder adalah satu kelas (huruf). Nama file berisi informasi latar dan HP yang dipakai, misalnya `A_Polos Gelap_Samsung_A55.jpg`. Informasi HP ini dipakai sebagai **grup** saat membagi data (penjelasannya di bagian 4).

In [ ]:
# Mengambil nama HP dari nama file, dipakai sebagai grup
def ambil_perangkat(nama_file):
    stem = Path(nama_file).stem
    stem = re.sub(r'\.jpe?g$', '', stem, flags=re.I)     # ada file bernama .jpg.jpg
    stem = re.sub(r'\(\d+\)', '', stem)                  # hapus penanda (2)
    bagian = [b.strip() for b in stem.split('_') if b.strip()]

    # file tanpa keterangan, misalnya 20240509_182720.jpg
    if len(bagian) <= 2 or not re.match(r'^[A-Za-z]$', bagian[0]):
        return 'tanpa_keterangan'

    perangkat = ' '.join(bagian[-2:]) if len(bagian) >= 4 else bagian[-1]
    perangkat = perangkat.lower().replace('apple', '').replace('background', '')
    perangkat = re.sub(r'[^a-z0-9]', '', perangkat)

    # menyamakan penulisan yang berbeda untuk HP yang sama
    perbaikan = {'iph15': 'iphone15', 'iphiphone15': 'iphone15', 'samsunga53': 'samsunga535g',
                 'redminote12peo': 'redminote12pro', 'note9': 'xiaomiredminote9',
                 'infinixnote30jpg': 'infinixnote30'}
    return perbaikan.get(perangkat, perangkat)

In [ ]:
# Membaca semua file citra
class_names = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir()])
ekstensi = {'.jpg', '.jpeg', '.png'}

data = []
for label, kelas in enumerate(class_names):
    for f in sorted((DATASET_DIR / kelas).iterdir()):
        if f.suffix.lower() in ekstensi and not f.name.startswith('.'):
            data.append({'path': str(f), 'file': f.name, 'kelas': kelas, 'label': label,
                         'perangkat': ambil_perangkat(f.name)})

df = pd.DataFrame(data)
num_classes = len(class_names)

print('Jumlah citra    :', len(df))
print('Jumlah kelas    :', num_classes)
print('Jumlah perangkat:', df['perangkat'].nunique())
df.head()

## 3. Mengecek Kualitas Data

In [ ]:
# Cek ukuran, mode warna, file rusak, dan file yang isinya sama persis (duplikat)
ukuran, mode, rusak, hash_file = Counter(), Counter(), [], {}
for p in df['path']:
    try:
        with Image.open(p) as im:
            ukuran[im.size] += 1
            mode[im.mode] += 1
    except Exception as e:
        rusak.append(p)
    hash_file.setdefault(hashlib.md5(open(p, 'rb').read()).hexdigest(), []).append(p)

duplikat = [v for v in hash_file.values() if len(v) > 1]
print('Ukuran citra :', dict(ukuran))
print('Mode warna   :', dict(mode))
print('File rusak   :', len(rusak))
print('Grup duplikat:', len(duplikat))

In [ ]:
# Jumlah citra per kelas
jumlah = df['kelas'].value_counts().sort_index()
jumlah.plot(kind='bar', figsize=(14, 4), color='#4c72b0', rot=0)
plt.title('Jumlah citra per huruf')
plt.ylabel('Jumlah citra')
plt.savefig(RESULT_DIR / 'distribusi_kelas.png', bbox_inches='tight')
plt.show()
print(jumlah.describe().round(1))

In [ ]:
# Contoh satu citra untuk setiap huruf
fig, axes = plt.subplots(3, 9, figsize=(18, 6.5))
for ax, kelas in zip(axes.ravel(), class_names):
    contoh = df[df['kelas'] == kelas].sample(1, random_state=SEED)['path'].iloc[0]
    ax.imshow(Image.open(contoh))
    ax.set_title(kelas)
    ax.axis('off')
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'contoh_citra.png', bbox_inches='tight')
plt.show()

In [ ]:
# Jumlah citra per perangkat, kebanyakan satu perangkat memotret 4 latar untuk setiap huruf
per_perangkat = df.groupby('perangkat').agg(jumlah_citra=('file', 'size'), jumlah_huruf=('kelas', 'nunique'))
per_perangkat.sort_values('jumlah_citra', ascending=False).head(10)

## 4. Membagi Data Latih, Validasi, dan Uji
Satu orang biasanya memotret huruf yang sama 4 kali dengan HP yang sama, hanya latarnya yang berbeda. Kalau data dibagi acak per foto, foto tangan orang yang sama bisa masuk ke data latih dan data uji sekaligus, sehingga model dinilai dengan tangan yang sudah pernah dilihatnya. Padahal di aplikasi nanti, model harus mengenali tangan orang baru.

Karena itu data dibagi dengan **StratifiedGroupKFold** berdasarkan perangkat: semua foto dari satu perangkat hanya masuk ke salah satu bagian. Komposisinya kurang lebih 70% latih, 15% validasi, dan 15% uji.

In [ ]:
# Bagian uji: 1 dari 7 lipatan grup (sekitar 15%)
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
idx_sisa, idx_test = next(sgkf.split(df, df['label'], df['perangkat']))

# Bagian validasi: 1 dari 6 lipatan dari sisa data (sekitar 15% dari total)
sisa = df.iloc[idx_sisa]
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
i_train, i_val = next(sgkf2.split(sisa, sisa['label'], sisa['perangkat']))
idx_train, idx_val = idx_sisa[i_train], idx_sisa[i_val]

df['split'] = ''
df.loc[idx_train, 'split'] = 'train'
df.loc[idx_val, 'split'] = 'validation'
df.loc[idx_test, 'split'] = 'test'

# Pastikan tidak ada perangkat yang muncul di dua bagian
p_train, p_val, p_test = (set(df.loc[i, 'perangkat']) for i in (idx_train, idx_val, idx_test))
assert not (p_train & p_val) and not (p_train & p_test) and not (p_val & p_test)

print(df['split'].value_counts())
print()
print('Perangkat di data uji:', sorted(p_test))
df.to_csv(RESULT_DIR / 'pembagian_dataset.csv', index=False)

In [ ]:
# Jumlah citra tiap huruf di setiap bagian
pd.crosstab(df['kelas'], df['split'])[['train', 'validation', 'test']].T

## 5. Pipeline Data
Data dibaca dengan `tf.data` langsung dari daftar file, jadi tidak perlu menyalin gambar ke folder lain. Contoh di materi memakai `ImageDataGenerator`, tetapi fungsi itu sudah tidak dipakai lagi di Keras 3, jadi augmentasi dibuat dengan layer Keras (`RandomFlip`, `RandomRotation`, dan lainnya) yang ikut menjadi bagian model. Augmentasi hanya aktif saat pelatihan.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Membaca satu gambar dan mengubah ukurannya
def baca_gambar(path, label, size):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (size, size))
    return img, label

# Membuat dataset dari indeks baris df
def buat_dataset(idx, size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values[idx], df['label'].values[idx]))
    if shuffle:
        ds = ds.shuffle(len(idx), seed=SEED)
    ds = ds.map(lambda p, l: baca_gambar(p, l, size), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Augmentasi data, mirip pengaturan ImageDataGenerator di materi tetapi rotasinya lebih kecil
def buat_augmentasi():
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal'),       # tangan kiri dan kanan
        layers.RandomRotation(0.1),            # sekitar 36 derajat
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.2),
    ], name='augmentasi')

In [ ]:
# Melihat hasil augmentasi pada satu gambar
contoh, _ = baca_gambar(df['path'].iloc[0], 0, MNV2_SIZE)
aug = buat_augmentasi()
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(contoh.numpy().astype('uint8')); axes[0].set_title('asli')
for ax in axes[1:]:
    ax.imshow(aug(contoh[None], training=True)[0].numpy().clip(0, 255).astype('uint8'))
    ax.set_title('augmentasi')
for ax in axes:
    ax.axis('off')
plt.show()

## 6. Fungsi Evaluasi
Sama seperti tugas-tugas sebelumnya, evaluasi memakai precision, recall, F1-score, dan akurasi. Karena ada 26 kelas, precision, recall, dan F1-score memakai rata-rata macro. Waktu pelatihan dan waktu prediksi juga dicatat. Waktu prediksi per gambar penting untuk aplikasi real-time.

In [ ]:
# Menghitung semua metrik evaluasi pada satu dataset
def evaluasi(model, ds, y_true):
    mulai = time.time()
    prob = model.predict(ds, verbose=0)
    waktu_prediksi = time.time() - mulai
    y_pred = prob.argmax(axis=1)
    return y_pred, {
        'Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Akurasi': accuracy_score(y_true, y_pred),
        'Waktu Prediksi (detik)': waktu_prediksi,
        'Waktu per Gambar (ms)': waktu_prediksi / len(y_true) * 1000,
    }

# Grafik akurasi dan loss selama pelatihan
def plot_history(histories, judul, nama_file):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history['accuracy']; val_acc += h.history['val_accuracy']
        loss += h.history['loss']; val_loss += h.history['val_loss']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(acc, label='Training Accuracy'); axes[0].plot(val_acc, label='Validation Accuracy')
    axes[1].plot(loss, label='Training Loss'); axes[1].plot(val_loss, label='Validation Loss')
    if len(histories) > 1:
        for ax in axes:
            ax.axvline(len(histories[0].history['loss']) - 0.5, color='gray', linestyle=':', label='mulai fine-tuning')
    for ax, t in zip(axes, ['Accuracy', 'Loss']):
        ax.set_title(f'{judul} {t}'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / nama_file, bbox_inches='tight')
    plt.show()

# Confusion matrix
def plot_cm(y_true, y_pred, judul, nama_file):
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    plt.title(judul); plt.xlabel('Prediksi'); plt.ylabel('Aktual')
    plt.savefig(RESULT_DIR / nama_file, bbox_inches='tight')
    plt.show()

In [ ]:
# Callback yang dipakai kedua model
# start_from_epoch: EarlyStopping baru mulai memantau setelah epoch tersebut, karena di awal pelatihan nilai validasi masih naik turun
def buat_callbacks(patience=8, start_from_epoch=0):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True,
                                         start_from_epoch=start_from_epoch, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]

y_val = df['label'].values[idx_val]
y_test = df['label'].values[idx_test]
hasil_holdout = {}

## 7. Model 1: CNN dari Awal (Baseline)
CNN ini dilatih tanpa bobot awal. Dibanding versi sebelumnya (`train_custom_cnn.py`), ada beberapa perbaikan:
- Augmentasi benar-benar dipasang ke model (sebelumnya didefinisikan tetapi tidak dipakai).
- Ditambah satu blok konvolusi dan Batch Normalization supaya pelatihan lebih stabil.
- Learning rate dinaikkan ke 0.001.

Model sebelumnya berhenti di akurasi 4%, sama dengan menebak acak dari 26 kelas, dan loss-nya tidak turun dari ln(26) = 3,258.

In [ ]:
# Arsitektur CNN dari awal
def buat_cnn():
    inputs = tf.keras.Input(shape=(CNN_SIZE, CNN_SIZE, 3))
    x = buat_augmentasi()(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='CNN_BISINDO')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn = buat_cnn()
cnn.summary()

In [ ]:
# Pelatihan CNN
train_cnn = buat_dataset(idx_train, CNN_SIZE, shuffle=True)
val_cnn = buat_dataset(idx_val, CNN_SIZE)
test_cnn = buat_dataset(idx_test, CNN_SIZE)

mulai = time.time()
hist_cnn = cnn.fit(train_cnn, validation_data=val_cnn, epochs=CNN_EPOCHS,
                   callbacks=buat_callbacks(patience=10, start_from_epoch=15), verbose=2)
waktu_latih_cnn = time.time() - mulai
print('Waktu pelatihan CNN: %.1f detik' % waktu_latih_cnn)

In [ ]:
plot_history([hist_cnn], 'CNN', 'cnn_training.png')

pred_cnn, hasil = evaluasi(cnn, test_cnn, y_test)
hasil['Waktu Pelatihan (detik)'] = waktu_latih_cnn
hasil['Jumlah Parameter'] = cnn.count_params()
hasil_holdout['CNN'] = hasil
print(classification_report(y_test, pred_cnn, target_names=class_names, zero_division=0))

In [ ]:
plot_cm(y_test, pred_cnn, 'Confusion Matrix CNN (data uji)', 'cnn_confusion_matrix.png')
cnn.save(MODEL_DIR / 'cnn_bisindo.keras')

## 8. Model 2: MobileNetV2 dengan Transfer Learning
Polanya mengikuti materi (InceptionV3, ResNet50, Xception): model dimuat dengan `include_top=False` dan `weights='imagenet'`, lalu ditambah layer klasifikasi di atasnya. MobileNetV2 dipilih karena modelnya kecil dan cepat, sehingga cocok untuk aplikasi web real-time.

Pelatihannya dua tahap:
1. **Tahap 1**: semua layer MobileNetV2 dibekukan, hanya layer klasifikasi baru yang dilatih (sama seperti di materi).
2. **Tahap 2 (fine-tuning)**: layer MobileNetV2 dibuka dan dilatih ulang dengan learning rate kecil (0.0001). Layer Batch Normalization tetap dibekukan supaya statistik ImageNet-nya tidak rusak.

Tahap 2 diperlukan karena fitur ImageNet dipelajari dari foto benda sehari-hari, bukan bentuk jari. Pada uji coba, model yang hanya melatih layer atas mentok di akurasi sekitar 50%.

Bedanya dengan materi: setelah MobileNetV2 dipakai `GlobalAveragePooling2D`, bukan `Flatten` + `Dense(512)`. Hasilnya parameter yang dilatih jauh lebih sedikit (sekitar 33 ribu dibanding 32 juta), sehingga risiko overfitting lebih kecil dan model lebih ringan untuk web.

In [ ]:
# Arsitektur MobileNetV2 + layer klasifikasi
def buat_mobilenetv2():
    base = tf.keras.applications.MobileNetV2(input_shape=(MNV2_SIZE, MNV2_SIZE, 3),
                                             include_top=False, weights='imagenet')
    base.trainable = False

    inputs = tf.keras.Input(shape=(MNV2_SIZE, MNV2_SIZE, 3))
    x = buat_augmentasi()(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)    # skala piksel ke -1 sampai 1
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='MobileNetV2_BISINDO')
    return model, base

# Tahap 1, hanya layer klasifikasi
def compile_tahap1(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Tahap 2, buka layer MobileNetV2 kecuali Batch Normalization
def compile_tahap2(model, base):
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

mnv2, base_mnv2 = buat_mobilenetv2()
compile_tahap1(mnv2)
mnv2.summary()

In [ ]:
# Pelatihan MobileNetV2
train_mnv2 = buat_dataset(idx_train, MNV2_SIZE, shuffle=True)
val_mnv2 = buat_dataset(idx_val, MNV2_SIZE)
test_mnv2 = buat_dataset(idx_test, MNV2_SIZE)

mulai = time.time()
hist_mnv2_1 = mnv2.fit(train_mnv2, validation_data=val_mnv2, epochs=MNV2_HEAD_EPOCHS, verbose=2)

compile_tahap2(mnv2, base_mnv2)
hist_mnv2_2 = mnv2.fit(train_mnv2, validation_data=val_mnv2, epochs=MNV2_FT_EPOCHS,
                       callbacks=buat_callbacks(patience=5), verbose=2)
waktu_latih_mnv2 = time.time() - mulai
print('Waktu pelatihan MobileNetV2: %.1f detik' % waktu_latih_mnv2)

In [ ]:
plot_history([hist_mnv2_1, hist_mnv2_2], 'MobileNetV2', 'mobilenetv2_training.png')

pred_mnv2, hasil = evaluasi(mnv2, test_mnv2, y_test)
hasil['Waktu Pelatihan (detik)'] = waktu_latih_mnv2
hasil['Jumlah Parameter'] = mnv2.count_params()
hasil_holdout['MobileNetV2'] = hasil
print(classification_report(y_test, pred_mnv2, target_names=class_names, zero_division=0))

In [ ]:
plot_cm(y_test, pred_mnv2, 'Confusion Matrix MobileNetV2 (data uji)', 'mobilenetv2_confusion_matrix.png')

## 9. Perbandingan Hasil pada Data Uji

In [ ]:
tabel_holdout = pd.DataFrame(hasil_holdout).T[['Precision', 'Recall', 'F1-Score', 'Akurasi',
                                                'Waktu Pelatihan (detik)', 'Waktu Prediksi (detik)',
                                                'Waktu per Gambar (ms)', 'Jumlah Parameter']]
tabel_holdout.round(4)

In [ ]:
# Grafik perbandingan metrik
tabel_holdout[['Precision', 'Recall', 'F1-Score', 'Akurasi']].astype(float).plot(kind='bar', figsize=(9, 4.5), rot=0)
plt.ylim(0, 1)
plt.title('Perbandingan CNN dan MobileNetV2 pada data uji')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=4)
plt.grid(axis='y', alpha=0.3)
plt.savefig(RESULT_DIR / 'perbandingan_model.png', bbox_inches='tight')
plt.show()

In [ ]:
# F1-score tiap huruf untuk kedua model, untuk melihat huruf yang sulit dikenali
f1_huruf = pd.DataFrame({
    'CNN': f1_score(y_test, pred_cnn, average=None, labels=range(num_classes), zero_division=0),
    'MobileNetV2': f1_score(y_test, pred_mnv2, average=None, labels=range(num_classes), zero_division=0),
}, index=class_names)
f1_huruf.plot(kind='bar', figsize=(15, 4), rot=0)
plt.title('F1-score tiap huruf (data uji)')
plt.grid(axis='y', alpha=0.3)
plt.savefig(RESULT_DIR / 'f1_per_huruf.png', bbox_inches='tight')
plt.show()
print('Huruf paling sulit untuk MobileNetV2:')
print(f1_huruf['MobileNetV2'].sort_values().head(5).round(3))

## 10. Eksperimen K-Fold Cross Validation
Sesuai materi, kedua model juga diuji dengan k-fold cross validation (k = 3). Pembagian fold juga berdasarkan perangkat (StratifiedGroupKFold). Di setiap fold, sebagian kecil data latih (juga berdasarkan perangkat) dipakai sebagai data validasi untuk EarlyStopping.

Bagian ini paling lama karena setiap model dilatih 3 kali. Kalau tidak ada GPU, `RUN_KFOLD` bisa diubah ke `False` dulu.

In [ ]:
hasil_kfold = []

if RUN_KFOLD:
    skf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=SEED)
    for fold, (idx_tr_all, idx_te) in enumerate(skf.split(df, df['label'], df['perangkat']), start=1):
        # sebagian data latih dipakai untuk validasi
        sub = df.iloc[idx_tr_all]
        i_tr, i_va = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
                          .split(sub, sub['label'], sub['perangkat']))
        idx_tr, idx_va = idx_tr_all[i_tr], idx_tr_all[i_va]
        y_te = df['label'].values[idx_te]

        # CNN
        tf.keras.backend.clear_session()
        model = buat_cnn()
        mulai = time.time()
        model.fit(buat_dataset(idx_tr, CNN_SIZE, shuffle=True), validation_data=buat_dataset(idx_va, CNN_SIZE),
                  epochs=KFOLD_CNN_EPOCHS, callbacks=buat_callbacks(patience=10, start_from_epoch=15), verbose=0)
        waktu = time.time() - mulai
        _, h = evaluasi(model, buat_dataset(idx_te, CNN_SIZE), y_te)
        hasil_kfold.append({'Algoritma': 'CNN', 'Eksperimen ke': fold, **h, 'Waktu Pelatihan (detik)': waktu})

        # MobileNetV2
        tf.keras.backend.clear_session()
        model, base = buat_mobilenetv2()
        compile_tahap1(model)
        tr_ds = buat_dataset(idx_tr, MNV2_SIZE, shuffle=True)
        va_ds = buat_dataset(idx_va, MNV2_SIZE)
        mulai = time.time()
        model.fit(tr_ds, validation_data=va_ds, epochs=MNV2_HEAD_EPOCHS, verbose=0)
        compile_tahap2(model, base)
        model.fit(tr_ds, validation_data=va_ds, epochs=KFOLD_MNV2_FT_EPOCHS, callbacks=buat_callbacks(patience=5), verbose=0)
        waktu = time.time() - mulai
        _, h = evaluasi(model, buat_dataset(idx_te, MNV2_SIZE), y_te)
        hasil_kfold.append({'Algoritma': 'MobileNetV2', 'Eksperimen ke': fold, **h, 'Waktu Pelatihan (detik)': waktu})

        print('Fold', fold, 'selesai')

    hasil_kfold = pd.DataFrame(hasil_kfold)

In [ ]:
# Tabel per fold ditambah rata-rata (format sama seperti tugas sebelumnya)
def tabel_kfold(nama):
    t = hasil_kfold[hasil_kfold['Algoritma'] == nama].set_index('Eksperimen ke')
    t = t[['Precision', 'Recall', 'F1-Score', 'Akurasi', 'Waktu Pelatihan (detik)', 'Waktu Prediksi (detik)']]
    t.loc['Rata-rata'] = t.mean()
    return t.round(4)

if RUN_KFOLD:
    print('Hasil Eksperimen CNN')
    display(tabel_kfold('CNN'))
    print('Hasil Eksperimen MobileNetV2')
    display(tabel_kfold('MobileNetV2'))

In [ ]:
# Rangkuman rata-rata k-fold
if RUN_KFOLD:
    rangkuman_kfold = hasil_kfold.groupby('Algoritma')[['Waktu Pelatihan (detik)', 'Waktu Prediksi (detik)',
                                                        'Precision', 'Recall', 'F1-Score', 'Akurasi']].mean()
    display(rangkuman_kfold.round(4))

## 11. Menyimpan Model dan Hasil
Model MobileNetV2 dari bagian 8 disimpan sebagai model untuk aplikasi web, bersama urutan nama kelas dan ukuran input yang dibutuhkan. Semua tabel hasil disimpan ke satu file Excel untuk bahan laporan.

In [ ]:
# Simpan model untuk aplikasi web
mnv2.save(MODEL_DIR / 'mobilenetv2_bisindo.keras')

# urutan nama kelas, indeks ke-i sama dengan output ke-i dari model
with open(MODEL_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=4)

# informasi input model untuk aplikasi web
info_model = {
    'model': 'mobilenetv2_bisindo.keras',
    'input_size': MNV2_SIZE,
    'input': 'gambar RGB dengan nilai piksel 0 sampai 255, preprocess_input sudah ada di dalam model',
}
with open(MODEL_DIR / 'model_info.json', 'w') as f:
    json.dump(info_model, f, indent=4)

# Simpan semua tabel hasil
with pd.ExcelWriter(RESULT_DIR / 'hasil_eksperimen.xlsx') as writer:
    tabel_holdout.round(4).to_excel(writer, sheet_name='Holdout')
    f1_huruf.round(4).to_excel(writer, sheet_name='F1 per Huruf')
    if RUN_KFOLD:
        hasil_kfold.round(4).to_excel(writer, sheet_name='KFold per Fold', index=False)
        rangkuman_kfold.round(4).to_excel(writer, sheet_name='KFold Rata-rata')
print('Model dan hasil tersimpan')

In [ ]:
# Uji cepat: memuat ulang model dan memprediksi satu gambar, seperti yang nanti dilakukan aplikasi web
model_web = tf.keras.models.load_model(MODEL_DIR / 'mobilenetv2_bisindo.keras')
contoh = df.iloc[idx_test].sample(1, random_state=SEED).iloc[0]
img = np.array(Image.open(contoh['path']).convert('RGB').resize((MNV2_SIZE, MNV2_SIZE)), dtype='float32')

model_web(img[None], training=False)          # pemanggilan pertama selalu lebih lama, jadi tidak dihitung
mulai = time.time()
prob = model_web(img[None], training=False).numpy()[0]
print('Waktu prediksi 1 gambar: %.1f ms' % ((time.time() - mulai) * 1000))
print('Label asli :', contoh['kelas'])
print('Prediksi   :', class_names[prob.argmax()], '(%.1f%%)' % (prob.max() * 100))